# Hybrid ALNS Execution and Benchmark Notebook

This notebook executes and benchmarks the metaheuristic after the offline model is trained.


## 1) Preconditions

- `repair_model.pkl` exists (trained in the training notebook).
- Benchmark instances are available in `../instances` (or your chosen folder).


## 2) Run baseline ALNS (no ML guidance)


In [ ]:
!python solver.py \
    --instances-dir ../instances \
    --iters 2000 \
    --alns \
    --seed 42 \
    --out baseline_results.csv


## 3) Run hybrid ALNS + offline repair model


In [ ]:
!python solver.py \
    --instances-dir ../instances \
    --iters 2000 \
    --alns \
    --ml-repair \
    --model-path repair_model.pkl \
    --seed 42 \
    --out hybrid_results.csv


## 4) Compare results

The next cell computes aggregate metrics and relative improvements.


In [ ]:
import pandas as pd

b = pd.read_csv('baseline_results.csv')
h = pd.read_csv('hybrid_results.csv')

key = ['instance'] if 'instance' in b.columns and 'instance' in h.columns else None
if key:
    m = b.merge(h, on=key, suffixes=('_baseline', '_hybrid'))
else:
    m = pd.concat([b.add_suffix('_baseline'), h.add_suffix('_hybrid')], axis=1)

summary = {}
for col in ['bins','objective','runtime_sec']:
    cb, ch = f'{col}_baseline', f'{col}_hybrid'
    if cb in m.columns and ch in m.columns:
        summary[col] = {
            'baseline_mean': m[cb].mean(),
            'hybrid_mean': m[ch].mean(),
            'relative_change_%': 100*(m[ch].mean()-m[cb].mean())/max(abs(m[cb].mean()),1e-9),
        }

pd.DataFrame(summary).T


## 5) Notes

- Lower bins/objective is better.
- Positive runtime change may be acceptable if solution quality improves.
